## Schema Writing

* Do not apply Not Null constraintor any constraint(becoz spark do not enforce it) in schema(DDL or struct type)
* Spark enforces Not Null only in delta file format but in that also don't mention Not Null in schema, handle it explicitly

In [0]:
# this string is case in-sensitive
DDL = """
    id int not null, 
    user_name string,
    abc long,
    salary FLOAT,
    net_worth DOUBLE,
    is_active BOOLEAN,
    join_date DATE,
    last_login TIMESTAMP,
    address STRUCT<street: STRING, city: STRING, zip: INT>,
    accuracy DECIMAL(10, 2)
"""
# Note - address column would look like JSON {"street": "123 Park Street", "city": "Kolkata", "zip": 700016}

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("id", IntegerType(), False),  # Default value -> True means allow NULL
    StructField("user_name", StringType()),
    StructField("abc", LongType()),
    StructField("salary", FloatType()),
    StructField("net_worth", DoubleType()),
    StructField("is_active", BooleanType()),
    StructField("join_date", DateType()),
    StructField("last_login", TimestampType()),
    StructField("address", StructType([
        StructField("street", StringType()),
        StructField("city", StringType()),
        StructField("zip", IntegerType())
    ])),
    StructField("accuracy", DecimalType(10, 2))
])

In [0]:
df.printSchema() # To print schema


**Spark Data Types: SQL vs. PySpark**

| Category | Spark SQL (DDL) | PySpark Class | Size | Range / Format | Key Point |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Numeric** | `BYTE`  | `ByteType()` | 1 Byte | -128 to 127 | Ideal for small flags/codes. |
| **Numeric** | `SHORT`  | `ShortType()` | 2 Bytes | -32,768 to 32,767 | Used for small number ranges. |
| **Numeric** | `INT`/`INTEGER`  | `IntegerType()` | 4 Bytes | -2.1B to 2.1B | Standard choice for IDs/counts. |
| **Numeric** | `LONG`  | `LongType()` | 8 Bytes | -9.2Q to 9.2Q | Use for massive counts/IDs. |
| **Numeric** | `FLOAT`  | `FloatType()` | 4 Bytes | 7 decimal digits | Faster, but lower precision. |
| **Numeric** | `DOUBLE` | `DoubleType()` | 8 Bytes | 15-17 decimal digits | Default for scientific decimals. |
| **Numeric** | `DECIMAL(p,s)` | `DecimalType()` | Variable | Up to 38 digits | **Best for money/finance.** |
| **String** | `STRING` | `StringType()` | Variable | UTF-8 Text | Variable length characters. |
| **Binary** | `BINARY` | `BinaryType()` | Variable | Byte array | For images, files, or blobs. |
| **Boolean** | `BOOLEAN` | `BooleanType()` | 1 Byte | `True` / `False` | Logical true/false values. |
| **Temporal** | `DATE` | `DateType()` | 4 Bytes | YYYY-MM-DD | No time or timezone info. |
| **Temporal** | `TIMESTAMP` | `TimestampType()` | 8 Bytes | YYYY-MM-DD HH:MM:SS | High-precision time with TZ. |
| **Complex** | `ARRAY<T>` | `ArrayType()` | Variable | `[val1, val2]` | All elements must be same type. |
| **Complex** | `MAP<K, V>` | `MapType()` | Variable | `{k1: v1, k2: v2}` | Keys/Values must be same type. |
| **Complex** | `STRUCT<...>` | `StructType()` | Variable | `(f1, f2, f3)` | Can hold **different** types. |



* Decimal Strictness: Always use `DECIMAL` for financial calculations. `FLOAT` and `DOUBLE` use "floating-point" math which can lead to tiny errors (like 0.00000001) that ruin accounting reports.
* Complex Types:
    * **Struct:** Fixed schema (like a table within a cell). Use when you know the fields (e.g., `address.city`).
    * **Map:** Flexible schema. Use when you don't know the keys in advance (e.g., `user_tags['dynamic_key']`).


## Data Read

#### Read CSV

spark.read -> format & header -> read mode -> inferschema or schema -> load
* Note - CSV and JSON are text files. They don't store metadata about data types. So, you should enable either inferschema option or provide schema manually, otherwise Spark will treat every single column as a StringType
* Read modes like FAILFAST, DROPMALFORMED, or PERMISSIVE are specific to file-based text formats (CSV and JSON) that require parsing. Don't use these read modes with parquet, avro & delta file.
In parquet, avro, delta file formats, corruption is handled differently: Because Parquet, Delta, and Avro are binary formats, "malformed" records usually mean the entire file is corrupted at the disk level. In these cases, Spark will simply throw an AnalysisException or a FileReadException immediately.

In [0]:
df = ( spark.read
      .format("csv")
      .option("header","true")  
      .option("mode","permissive")  # permissive(default), dropMalformed, failFast (Note- these strings are case insensitive)
      .option("inferschema","true") # Default value is False
      .load("path")
)

In [0]:
df = (spark.read
      .format("csv")
      .option("header","true")  # you can also write True
      .option("mode","failFast") # can also write failfast
      .schema(my_schema)
      .load("path")
)


#### Read Parquet, Avro, Delta

spark.read -> format -> load<br>
* Parquet, Avro, Delta are self-describing formats means they store the schema. So, don't provide inferschema option as true, even if u provide the option won't do anything. 
* Don't provide schema manually while reading delta file format. Delta file stores schema in delta log. If you try to provide a schema that conflicts with the Delta log, Spark will usually ignore your manual schema and stick to the log's definition to ensure data integrity.
* You can provide a manual schema while reading parquet or avro file format. In this case Spark will ignore the stored schema & will enforce the provided schema.

In [0]:
df = (
    spark.read
        .format("delta") # avro, parquet
        .load("path")
)

* Note - mergeschema option is only a feature in parquet & delta file format, you cannot use it with CSV, JSON, Avro.
* With parquet file format, we can only use mergeschema option during data reading and not during data write.<br>
Note - We use mergeSchema option when we have to read multiple Parquet files from a single directory and they have different schema.<br>
The Benefit:-<br>
It forces Spark to scan the footers of all files to create a unified schema. Without it, Spark might only read the schema from a single file, causing it to "miss" columns that exist in other files or throw errors when it encounters unexpected data.
* With delta file format we can only use mergeschema option during write operation.
* Delta lake validates the schema during write operation(schema on write) means it matches the incoming schema with the existing schema stored in delta log & if schema matches then only write operation happens otherwise it will throw error.
* Parquet, Avro, CSV, JSON donot enforce schema on write since they don't have delta log. Means you can drop a Parquet file with 5 columns into a folder, and then drop another Parquet file with 10 columns into that same folder. The storage system won't stop you.

In [0]:
df = (
    spark.read
        .format("parquet") 
        .option("mergeSchema", "true") # only for parquet & while reading multiple files
        .load("path")
)

#### Read JSON

spark.read -> format -> read mode -> inferschema or schema -> multiLine -> load

In [0]:
# without providing schema explicitly 
df = (spark.read
      .format("json")
      .option("mode", "FAILFAST")
      .option("inferSchema", "true") 
      .option("multiLine", "true") # By default -> False
      .load("path/to/json_files/")
      )

In Spark, the **`multiLine`** option determines how the JSON reader parses files. By default, Spark expects **JSON Lines** (one JSON object per line), but many APIs or systems provide **Standard/Pretty JSON** where a single object or array spans multiple lines.


 1. The Default: Single-Line JSON (`multiLine=false`)
Spark's default behavior is highly optimized for performance. It treats **each line** of the file as an independent, complete JSON record. 

Format Example:
```json
{"id": 1, "name": "Amit", "city": "Kolkata"}
{"id": 2, "name": "Sneha", "city": "Bangalore"}
```
* Performance: Very high. Spark can split this file and process different lines on different executors simultaneously.



2. The Alternative: Multi-Line JSON (`multiLine=true`)
If your JSON looks like a "pretty-printed" file or a single large array, Spark will fail to read it using the default settings because it sees a `[` or `{` on line 1 and expects the record to end there.

Format Example:
```json
[
  {
    "id": 1,
    "name": "Amit",
    "city": "Kolkata"
  },
  {
    "id": 2,
    "name": "Sneha",
    "city": "Bangalore"
  }
]
```

**Comparison: Single-Line vs. Multi-Line**

| Feature | Single-Line (Default) | Multi-Line (`true`) |
| :--- | :--- | :--- |
| **Structure** | One object per line. | One object/array across many lines. |
| **Read Speed** | **Fast.** Can be read in parallel. | **Slower.** Usually requires a single executor to parse the whole file. |
| **Splittability** | Yes (Spark can split large files). | No (The whole file must be read as one block). |
| **Memory** | Low footprint. | Can cause `OutOfMemory` errors if the file is massive. |

In [0]:
# Providing Schema explicitly
# JSON :-
{"id": 1, "name": "Amit", "dept": "Data Engineering", "salary": 85000.50}
{"id": 2, "name": "Sneha", "dept": "Analytics", "salary": 92000.00}

In [0]:
employee_schema = "id INT, name STRING, dept STRING, salary DOUBLE"
df = (spark.read
      .format("json")
      .option("mode", "FAILFAST")
      .schema(employee_schema)
      .load("path to file.json"))  # multiline is false by default

#### Read Streaming Data

In [0]:
# refer streaming part after interview
(spark.readStream
    .format("csv")
    .schema(my_schema) # don't provide schema in case of delta file
    .load("path")
)

#### Autoloader

In [0]:
df = (spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "json")          
          .option("cloudFiles.schemaLocation", “abfss: …….”) 
          .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # addNewColumns(Default), failOnNewColumns, rescue, none     
          .option("cloudFiles.inferColumnTypes", "true") 
          .load("/mnt/raw/input/"))                 


In [0]:
df = (spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "json")          
          .option("cloudFiles.schemaLocation", “abfss: …….”) 
          .option("cloudFiles.schemaEvolutionMode", "addNewColumns")    
          .schema(my_schema)
          .load("/mnt/raw/input/"))                  

#### Read Streaming JSON from Kafka

In [0]:
df = (spark.readStream
          .format("kafka")
          .options(**kafka_options)
          .load()
          )

# Parsing the JSON data
df = df.selectExpr("CAST(value AS STRING)") 
df = df.select(from_json(col("value"), schema).alias("data"))
df = df.select("data.*")


 **What is happening in this logic?**

1.  **`selectExpr("CAST(value AS STRING)")`**: When reading from sources like **Kafka** or **Event Hubs**, the `value` column is often in `Binary` format. This line converts it into a readable string so it can be parsed.
2.  **`from_json(col("value"), schema)`**: This takes that string and maps it against a predefined `schema` (StructType). It turns the string into a complex **Struct** object named `data`.
3.  **`.select("data.*")`**: This is the "flattening" step. It takes all the fields inside the `data` struct and turns them into individual top-level columns in your DataFrame.




## Data Write

df.write -> format -> mode -> partition by -> save
* Write mode & partition by is compatible with all file formats i.e. parquet, avro, csv, json, delta
* Bucketing can be used only with tables. Bucketing requires saving the data as a managed or external table in the Hive Metastore/Unity Catalog.
* Delta file format supports partitionBy, it does not support bucketBy. Instead, Delta uses a much more advanced feature called Z-Ordering, which organizes data within the files themselves without needing a fixed number of buckets.

In [0]:
(df.write
  .format("parquet")  # also use with delta, avro, csv, json
  .mode("overwrite")  # append, overwrite, ignore, error(default)
  .partitionBy("year","month")
  .save("path"))

In [0]:
(df.write
  .format("parquet") # Note - Bucket by doesn't work with delta
  .mode("overwrite")
  .partitionBy("join_date")          # Creates folders by date
  .bucketBy(10, "country","state")   # Creates 10 buckets based on ID
  .saveAsTable("employee_table")
  )

#### Write Streaming data

In [0]:
(
    df.writeStream
        .format("delta")
        .outputMode("Complete")  # Complete, Append(default), Update
        .option("checkpointLocation","...path...")
        .trigger(processingTime="5 seconds")  # or once = True
        .option("path","__path__")
        .start()
)

## Transformations

* expr() (short for expression) is a powerful function that allows you to write SQL-like strings and have them treated as programmatic column operations.
* We can use expr() in almost all the methods that are designed to take Column objects as inputs.

`selectExpr()` is a highly efficient variant of the `select()` method that allows you to perform column selection and complex SQL transformations in a single step.

While `select()` usually takes column objects or names, `selectExpr()` specifically expects **SQL strings**.


##### 1. Why use `selectExpr()`?
In standard PySpark, if you want to select a column and rename it or modify it, you often have to chain multiple methods like `.select()`, `.withColumn()`, and `.withColumnRenamed()`. 

`selectExpr()` combines all of these into one readable string.

**Example:** The Old Way vs. The `selectExpr` Way<br>
Imagine you want to select `name` in uppercase, calculate a `10% tax` on salary, and keep the `id`.

Standard PySpark:
```python
from pyspark.sql.functions import col, upper

df.select(
    col("id"),
    upper(col("name")).alias("name_upper"),
    (col("salary") * 0.1).alias("tax")
)
```

Using selectExpr():
```python
df.selectExpr(
    "id", 
    "upper(name) as name_upper", 
    "salary * 0.1 as tax"
)
```



##### 2. Key Capabilities
* **Aliasing:** Use the `AS` keyword directly in the string to rename columns.
* **Built-in SQL Functions:** Use any Spark SQL function (like `avg()`, `sum()`, `coalesce()`, `lower()`) without importing them from `pyspark.sql.functions`.
* **Aggregate Functions:** You can use it to perform global aggregations, though it's less common than using `groupBy`.
    * *Example:* 
`df_summary = df.selectExpr("avg(salary) as avg_sal", "count(*) as total_employees")`
* **Star Selection:** You can use `*` to select all existing columns and add new ones simultaneously.
    * *Example:* `df.selectExpr("*", "salary + bonus as total_pay")`



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

dbuitls.fs.ls('directory_path')

df.display()
df.show()

In [0]:
df.count()
df.collect()
df.distinct()

In [0]:
df.select(col("col_A").alias("col_a"), col("col_B").alias("col_b"))
df.selectExpr("col_A as col_a", "col_B as col_b")
df.select(expr("col_A as col_a"), expr("col_B as col_b"))

In [0]:
df.filter(( col("col_A").isNull() ) & ( col("col_B") == 20 ) | ( col("col_C").isin('a','b','c') ))

df.filter("col_A IS NULL AND col_B >= 20 OR col_C IN ('a', 'b', 'c')")
# When you pass a string into .filter(), Spark automatically treats it as a SQL WHERE clause.
# If your Python string uses double quotes ("), use single quotes (') for values inside the SQL (e.g., 'Kolkata').
# We can use standard SQL operators like AND, OR, NOT, IN, IS NULL, and BETWEEN.

df.filter(expr("col_A IS NULL AND col_B >= 20 OR col_C IN ('a', 'b', 'c')"))

In [0]:
# filter & where works exactly the same
df.where(( col("col_A").isNull() ) & ( col("col_B") == 20 ) | ( col("col_C").isin('a','b','c') ))

df.where("col_A IS NULL AND col_B >= 20 OR col_C IN ('a', 'b', 'c')")

df.where(expr("col_A IS NULL AND col_B >= 20 OR col_C IN ('a', 'b', 'c')"))

In [0]:
df.withColumn("processed_time", current_timestamp())
df.withColumn('curr_date',current_date())
df.withColumn("id", monotonically_increasing_id())

df.withcolumn("flag",lit("new"))
df.selectExpr("*", "'new' as flag")

In [0]:
df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))

# The "*" selects everything, then we overwrite Item_Fat_Content
# Note: If you select a column name that already exists, the new one will replace the old one
df.selectExpr("*", "regexp_replace(Item_Fat_Content, 'Regular', 'Reg') as Item_Fat_Content")

df.withColumn("Item_Fat_Content", expr("regexp_replace(Item_Fat_Content, 'Regular', 'Reg')"))

In [0]:
df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))

# Overwrites Item_Weight with its String version
df.selectExpr("*", "cast(Item_Weight as string)")

df.withColumn("Item_Weight", expr("cast(Item_Weight as string)"))

In [0]:
df.orderBy(col("product_id").asc(), col("date").desc())

df.orderBy(expr("product_id asc"), expr("date desc"))
# orderBy & sort works exactly the same
df.sort(col("product_id").asc(), col("date").desc())
df.sort(expr("product_id asc"), expr("date desc"))

In [0]:
df.limit(10)

df.drop('Item_Visibility').display()  # will drop 1 column
df.drop('Item_Visibility','Item_Type').display()  # will drop 2 columns

In [0]:
df1.union(df2)

In [0]:
df.withColumn('week_after',date_add('curr_date',7))
df.withColumn('week_before',date_sub('curr_date',7))
df.withColumn('date_diff',datediff('end_date','start_date'))

In [0]:
df.dropDuplicates()
df.distinct()
df.dropDuplicates(subset=["product_id"])

In [0]:
df.dropna('all').display() 
df.dropna('any').display() 
df.dropna(subset=['Outlet_Size'])

In [0]:
# Calculate Average
avg_score = df.select(avg("Score")).collect()[0][0]

# Calculate Median
median_score = df.select(median("Score")).collect()[0][0]

# Calculate Mode
mode_city = df.select(mode("City")).collect()[0][0]

# Now apply them all in fillna
df = df.fillna({
    'Category': 'NA', 
    'Revenue': 0,
    'Score': avg_score,
    'City': mode_city
})

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1])
df.withColumn('Outlet_Type',explode('Outlet_Type'))

In [0]:
df.groupBy('Item_Type', 'Outlet_Size').agg(
    sum(col('Item_MRP')).alias('Total_MRP'),
    avg(col('Item_MRP')).alias('Average_MRP'),
    count('*').alias('Item_Count')
)

df.groupBy('Item_Type', 'Outlet_Size').agg(
    expr("sum(Item_MRP) as Total_MRP"),
    expr("avg(Item_MRP) as Average_MRP"),
    expr("count(*) as Item_Count")
)

In [0]:
df.withColumn('veg_exp_flag',when(((col('veg_flag')=='Veg') & (col('Item_MRP')<100)),'Veg_Inexpensive')\
                .when((col('veg_flag')=='Veg') & (col('Item_MRP')>100),'Veg_Expensive')\
                .otherwise('Non_Veg'))

df.withColumn("veg_exp_flag", expr("""
    CASE 
        WHEN veg_flag = 'Veg' AND Item_MRP < 100 THEN 'Veg_Inexpensive'
        WHEN veg_flag = 'Veg' AND Item_MRP > 100 THEN 'Veg_Expensive'
        ELSE 'Non_Veg'
    END
"""))

df.selectExpr("*", """
    CASE 
        WHEN veg_flag = 'Veg' AND Item_MRP < 100 THEN 'Veg_Inexpensive'
        WHEN veg_flag = 'Veg' AND Item_MRP > 100 THEN 'Veg_Expensive'
        ELSE 'Non_Veg'
    END as veg_exp_flag
""")

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'inner')
df1.join(df2,df1['dept_id']==df2['dept_id'],'left')
df1.join(df2,df1['dept_id']==df2['dept_id'],'right')

In [0]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy("product_id").orderBy(col("date").asc()).rowsBetween(Window.unboundedPreceding, Window.currentRow)
df = df.withColumn("Cumulative_Sales", sum(col("sales")).over(window_spec))

df = df.withColumn("Cumulative_Sales", expr("""
    sum(sales) OVER (
        PARTITION BY product_id 
        ORDER BY date ASC 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )
"""))

In [0]:
df.createTempView('my_view')

%sql
select * from my_view where Item_Fat_Content = 'Lf'

df = spark.sql("select * from my_view where Item_Fat_Content = 'Lf'")

In [0]:
df.groupBy(spark_partition_id()).count().displa()

In [0]:
# Example: value counts for column "country"
df.groupBy("country").count().display

#### Merge

In [0]:
delta_tbl.alias('trg').merge(df.alias('src'),"src.id == trg.id")\
    .whenNotMatchedInsertAll()\
    .whenMatchedUpdateAll()\
    .execute()

In [0]:
MERGE INTO target_table AS target
USING source_table AS source
ON target.id = source.id
WHEN MATCHED THEN 
  UPDATE SET *
WHEN NOT MATCHED THEN 
  INSERT *

#### Windowing & Watermark - Spark Streaming

In [0]:
a = (df.withWatermark("event_time", "5 minutes") 
    .groupBy(
        window(col("event_time"), "10 minutes","8 minutes"), col("ward_id")
    )
    .agg(count("*").alias("admission_count"))
)

a.stop()

#### Views

In [0]:
df.createOrReplaceTempView("my_temp_view")
SELECT * FROM my_temp_view

df.createOrReplaceGlobalTempView("my_global_view")
SELECT * FROM global_temp.my_global_view


In [0]:
%sql
-- Standard SQL View
CREATE OR REPLACE VIEW my_catalog.my_schema.my_view
AS
SELECT * FROM my_table;


#### Delta Table

In [0]:
%sql
DESCRIBE HISTORY catalog.schema.table_name; 

RESTORE catalog.schema.table_name TO VERSION AS OF 3; 

VACUUM my_table;   -- Uses default retention (7 days)
VACUUM my_table RETAIN 8 DAYS;  -- Custom retention (e.g., 8 days)
VACUUM my_table RETAIN 7 HOURS;  -- Custom retention (e.g., 7 hours)
VACUUM my_table RETAIN 0 HOURS;  -- will delete all tombstoned file

OPTIMIZE catalog.schema.my_table;

OPTIMIZE my_table 
ZORDER BY (customer_id, order_date);

In [0]:
from delta.tables import DeltaTable
if DeltaTable.isDeltaTable(spark, watermark_path):
    print("Table exists, proceeding with update.")
else:
    print("Path is empty or not Delta. Initializing table...")

In [0]:
target_patient_tbl = DeltaTable.forPath(spark, dim_patient_path)

#### SCD Type 2

In [0]:
df_silver = spark.read

# Initial Run
if not DeltaTable.isDeltaTable(spark, path):
  # df.silver -> add surrogate key & active flag
  
df_gold = spark.read

# create hash value using sha2(concat_ws("||",col("column_name").cast("string"),....),256)

# find changed records
# df_changed = inner join df_silver & df_gold on id -> filter: active flag = True and hash_value is not equal -> select suurogate key from df_gold
# update
target_tbl.merge(
    source = df_changed,
    condition = target_tbl["surrogate_key"] == df_changed["surrogate_key"]
).whenMatchedUpdate(set = {"active": "False"})
.execute()

# Insert new and changed records
# df_insert = left join df_silver & df_gold on id -> filter: df_gold id is Null or (active flag = True and hash_value is not equal)
# df_insert -> add surrogate key & active flag -> True then insert


#### SCD Type 1

In [0]:
df_silver = spark.read

# Initial Run
if not DeltaTable.isDeltaTable(spark, path):
  # df.silver -> add surrogate key & insert
  
df_gold = spark.read

df = Left join df_silver & df_gold on id 
# df_new -> from df filter rows where df_gold.id is null
# df_existing -> from df filter rows where df_gold.id is not null

# add surrogate key in df_new
df_insert = df_new.union(df_existing)

delta_tbl = DeltaTable.forPath(spark, "abfss://gold@cardata1.dfs.core.windows.net/branch")
    
    delta_tbl.alias("t").merge(df_final.alias("s"), "s.branch_id = t.branch_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()



## DLT

In [0]:
import dlt

@dlt.table
@dlt.expect_or_drop("valid_age", "age > 0")                   # Drop invalid ages
@dlt.expect_or_fail("valid_id", "id IS NOT NULL")             # Fail if id is missing
@dlt.expect_all({"valid_email": "email LIKE '%@%'"})          # Log invalid emails but keep rows
def streaming_orders():
        df = spark.readStream.format("cloudFiles")      # Streaming table
        .option("cloudFiles.format", "json")
        .load("/mnt/raw/orders")
    return df


In [0]:
@dlt.table
def my_materialized_table():
    return spark.read.format("csv").option("header", True).load("/mnt/raw/data")


In [0]:
@dlt.view
def streaming_orders_view():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .load("/mnt/raw/orders")
    )

In [0]:
@dlt.view
def orders_temp_view():
    return spark.read.format("csv").option("header", True).load("/mnt/raw/orders")


In [0]:
import dlt

# Create an empty target streaming table
dlt.create_streaming_table("dim_customers_type1")

dlt.apply_changes(
    target = "dim_customers_type1",
    source = "silver_customers",
    keys = ["id"],
    sequence_by = col("date"),
    except_column_list = ["date","action"],
    apply_as_deletes = expr("action = 'd'"),
    stored_as_scd_type = 1
)
